In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/model.safetensors.index.json
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/config.json
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/model-00001-of-00002.safetensors
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/Responsible-Use-Guide.pdf
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/model-00002-of-00002.safetensors
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/pytorch_model-00002-of-00002.bin
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/README.md
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/USE_POLICY.md
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/tokenizer.json
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/tokenizer_config.json
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/pytorch_model.bin.index.json
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/LICENSE.txt
/kaggle/input/models/metaresearch/ll

In [2]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d not in (".git", "plots", "raw_plots")]
    for f in files:
        if f.endswith((".csv", ".md")): print(os.path.join(root, f))

/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/README.md
/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1/USE_POLICY.md
/kaggle/input/datasets/kushalag21/meld-b3/meld_wada_snr_final.csv
/kaggle/input/datasets/kushalag21/meld-b3/B3_meld/meld_processed_5class_dev_audio_features.csv
/kaggle/input/datasets/kushalag21/meld-b3/B3_meld/meld_processed_5class_test_audio_features.csv
/kaggle/input/datasets/kushalag21/meld-b3/B3_meld/meld_processed_5class_train_audio_features.csv


In [3]:
import time; T0 = time.time()
!pip install -q deepspeed bitsandbytes peft opensmile
!pip uninstall -y -q torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import os, re, json, glob, shutil, subprocess
!git clone -q https://github.com/zehuiwu/SpeechCueLLM /kaggle/working/SpeechCueLLM
os.chdir("/kaggle/working/SpeechCueLLM/LLM_code")

In [5]:
MODE = "B3"      # "B3" = denoised + SNR-aware | "B2" = denoised | "RAW_DESC" | "TEXT"

if MODE in ("B2", "B3"):
    key = "/meld-denoised-fixed/" if MODE == "B2" else "/meld-b3/B3_meld/"
    for s in ["train", "dev", "test"]:
        hits = [h for h in glob.glob(f"/kaggle/input/**/meld_processed_5class_{s}_audio_features.csv", recursive=True)
                if key in h]
        if len(hits) > 1:
            hits = [h for h in hits if "/SpeechCueLLM/" not in h]
        print(s, hits)
        assert len(hits) == 1, f"expected 1 file for {s}, got {len(hits)}"
        shutil.copy(hits[0], f"/kaggle/working/SpeechCueLLM/speech_features/meld_processed_5class_{s}_audio_features.csv")

import pandas as pd
PLACEHOLDER = "Target speech characteristics: not available."

if MODE != "TEXT":
    for s in ["train", "dev", "test"]:
        p = f"/kaggle/working/SpeechCueLLM/speech_features/meld_processed_5class_{s}_audio_features.csv"
        x = pd.read_csv(p, dtype=str, keep_default_na=False)
        desc = x["description"]
        bad = (~desc.str.contains(": ", regex=False)) | desc.str.startswith("Insufficient")
        print(s, x.shape, "| descriptions replaced:", int(bad.sum()))
        x.loc[bad, "description"] = PLACEHOLDER
        x.to_csv(p, index=False)
        chk = pd.read_csv(p, dtype=str, keep_default_na=False)["description"]
        assert chk.str.contains(": ", regex=False).all() and not chk.str.startswith("Insufficient").any(), f"bad rows left in {s}"

d = pd.read_csv("/kaggle/working/SpeechCueLLM/speech_features/meld_processed_5class_train_audio_features.csv")
print(MODE, d.shape, "| insufficient-data rows:", d["description"].str.startswith("Insufficient").sum())
print(d["description"].iloc[0])

train ['/kaggle/input/datasets/kushalag21/meld-b3/B3_meld/meld_processed_5class_train_audio_features.csv']
dev ['/kaggle/input/datasets/kushalag21/meld-b3/B3_meld/meld_processed_5class_dev_audio_features.csv']
test ['/kaggle/input/datasets/kushalag21/meld-b3/B3_meld/meld_processed_5class_test_audio_features.csv']
train (9988, 24) | descriptions replaced: 0
dev (1112, 24) | descriptions replaced: 0
test (2615, 24) | descriptions replaced: 0
B3 (9988, 24) | insufficient-data rows: 0
Target speech characteristics (estimated SNR 15 dB, moderate audio reliability): likely very high volume with moderate variation, likely low pitch with moderate variation, likely moderate speaking rate.


In [6]:
import os, glob

MODEL_PATH = "/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1"
EPOCHS = 3
DATASET, LR = "meld", "2e-4"            # the paper's MELD learning rate

print(sorted(os.listdir(MODEL_PATH)))
assert os.path.exists(MODEL_PATH + "/config.json"), "config.json missing: check the path"
assert glob.glob(MODEL_PATH + "/*.safetensors") or glob.glob(MODEL_PATH + "/*.bin"), "no weight files found"
assert os.path.exists(MODEL_PATH + "/tokenizer.model") or os.path.exists(MODEL_PATH + "/tokenizer.json"), "no tokenizer found"

['.gitattributes', 'LICENSE.txt', 'README.md', 'Responsible-Use-Guide.pdf', 'USE_POLICY.md', 'added_tokens.json', 'config.json', 'generation_config.json', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'pytorch_model-00001-of-00002.bin', 'pytorch_model-00002-of-00002.bin', 'pytorch_model.bin.index.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json']


In [7]:
s = open("main.py").read()
s = s.replace(
"        model =AutoModelForCausalLM.from_pretrained(args.model_name_or_path).half()",
"""        import time, gc, torch as _t
        if int(os.environ.get("LOCAL_RANK", 0)) == 1:
            time.sleep(180)
        from transformers import BitsAndBytesConfig
        _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=_t.float16)
        model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, quantization_config=_bnb,
                    torch_dtype=_t.float16, low_cpu_mem_usage=True,
                    device_map={"": int(os.environ.get("LOCAL_RANK", 0))})
        gc.collect()
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=bool(args.gradient_checkpointing))""")
s = s.replace('deepspeed_config["bfloat16"]["enabled"] = True\n        deepspeed_config["fp16"]["enabled"] = False',
              'deepspeed_config["bfloat16"]["enabled"] = False\n        deepspeed_config["fp16"]["enabled"] = True')
s = s.replace("    return optimizer_grouped_parameters",
              "    return [g for g in optimizer_grouped_parameters if len(g['params']) > 0]")
s = s.replace("    deepspeed_config[\"zero_optimization\"]['mics_shard_size'] = world_size", "    pass")
open("main.py", "w").write(s)

p = "data_utils/deepspeed_config.json"
cfg = json.load(open(p)); cfg.pop("amp", None); cfg["zero_optimization"]["stage"] = 2
json.dump(cfg, open(p, "w"), indent=2)
print("4bit", "BitsAndBytesConfig" in s, "| fp16", 'deepspeed_config["fp16"]["enabled"] = True' in s, "| amp removed", "amp" not in cfg)

4bit True | fp16 True | amp removed True


In [8]:
FLAGS = {"B2": ("True", "True"), "B3": ("True", "True"), "RAW_DESC": ("True", "True"), "TEXT": ("False", "False")}
desc, ctx = FLAGS[MODE]
sh = open("train_and_inference.sh").read()
def setv(name, val):
    global sh; sh = re.sub(rf"^{name}=.*$", f"{name}={val}", sh, count=1, flags=re.M)
setv("MODEL_NAME", "'LLaMA2'"); setv("Experiments_setting", "'lora'"); setv("dataset", f"'{DATASET}'")
setv("audio_description", f"'{desc}'"); setv("audio_context", f"'{ctx}'")
setv("audio_impression", "'False'"); setv("audio_only", "'False'")
setv("SEED", 11); setv("num_train_epochs", EPOCHS); setv("LORA_LR", LR)
setv("accumulations", 4); setv("graphics_card", 2); setv("data_percent", 1.0)
sh = sh.replace("MODEL_PATH='LLaMA2 MODELPATH'", f"MODEL_PATH='{MODEL_PATH}'")
sh = sh.replace("--seed ${SEED}\n    fi", "--seed ${SEED} \\\n        --gradient_checkpointing\n    fi")
open("train_and_inference.sh", "w").write(sh)
!grep -n -E "^(audio_|dataset|num_train|data_percent|SEED|LORA_LR)" train_and_inference.sh

21:dataset='meld'
25:audio_description='True'
26:audio_impression='False'
27:audio_context='True'
28:audio_only='False'
31:SEED=11
32:num_train_epochs=3
33:LORA_LR=2e-4
55:data_percent=1.0


In [9]:
LOG = f"/kaggle/working/run_log_MELD_{MODE}_7B_qlora.txt"
proc = subprocess.Popen(f"PYTHONUNBUFFERED=1 bash train_and_inference.sh > {LOG} 2>&1", shell=True)
print("started, pid", proc.pid)

started, pid 138


In [10]:
DEADLINE_H = 11.0

def status():
    lines = open(LOG, errors="ignore").read().replace("\r", "\n").split("\n")
    steps = [l for l in lines if "Running Loss" in l]
    last = steps[-1].strip()[-140:] if steps else "(loading model / not started)"
    f1 = [l for l in lines if l.startswith(("current_f1", "best_f1", "history"))][-3:]
    err = [l for l in lines if "Traceback" in l or "OutOfMemory" in l or "Killed" in l][-2:]
    gpu = subprocess.run("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " | ")
    ram = subprocess.run("free -g | awk 'NR==2{print $3\"/\"$2\" GB\"}'", shell=True, capture_output=True, text=True).stdout.strip()
    print(f"[{time.strftime('%H:%M:%S')} +{(time.time()-T0)/3600:.1f}h] {last}\n  GPU: {gpu} | RAM: {ram}")
    for l in f1 + err: print("  ", l.strip()[:200])

while proc.poll() is None:
    status(); time.sleep(300)
    if time.time() - T0 > DEADLINE_H * 3600:
        print("DEADLINE reached, stopping training"); subprocess.run("pkill -f main.py", shell=True); break
status()
subprocess.run("pkill -f main.py; pkill -f deepspeed", shell=True)
print("FINISHED, exit code", proc.poll())

if proc.poll() != 0:
    L = open(LOG, errors="ignore").read().replace("\r", "\n").split("\n")
    idx = [i for i, l in enumerate(L) if "Traceback" in l]
    s0 = max(0, (idx[0] if idx else len(L)) - 5)
    print("\n".join(l[:250] for l in L[s0:s0 + 80]))

[11:33:00 +0.0h] (loading model / not started)
  GPU: 0 MiB, 0 % | 0 MiB, 0 % | RAM: 1/31 GB
[11:38:00 +0.1h] (loading model / not started)
  GPU: 3763 MiB, 2 % | 3 MiB, 0 % | RAM: 3/31 GB
[11:43:00 +0.2h] Epochs 0/3. Running Loss:    2.6206:   2%|▏         | 120/4990 [02:31<1:34:40,  1.17s/it]
  GPU: 5331 MiB, 100 % | 5251 MiB, 100 % | RAM: 5/31 GB
[11:48:00 +0.3h] Epochs 0/3. Running Loss:    0.3822:   7%|▋         | 330/4990 [07:31<1:51:06,  1.43s/it]
  GPU: 5331 MiB, 100 % | 5251 MiB, 100 % | RAM: 5/31 GB
[11:53:00 +0.3h] Epochs 0/3. Running Loss:    0.1436:  11%|█         | 543/4990 [12:31<1:47:50,  1.46s/it]
  GPU: 5331 MiB, 100 % | 5251 MiB, 100 % | RAM: 5/31 GB
[11:58:00 +0.4h] Epochs 0/3. Running Loss:    0.1590:  15%|█▌        | 755/4990 [17:31<1:44:55,  1.49s/it]
  GPU: 5331 MiB, 100 % | 5251 MiB, 100 % | RAM: 5/31 GB
[12:03:00 +0.5h] Epochs 0/3. Running Loss:    1.3057:  19%|█▉        | 969/4990 [22:32<1:40:35,  1.50s/it]
  GPU: 5331 MiB, 95 % | 5251 MiB, 93 % | RAM: 5/31 G